# Evaluete wtq test

## Functions

In [10]:
import sys
import os
import pandas as pd
import numpy as np
# Добавляем корневую директорию проекта в sys.path
sys.path.append(os.path.dirname(os.path.abspath('/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/tests')))
from utils.utils import load_config
from datasets import load_from_disk,Dataset

In [7]:
def evaluate(res,gt:str):
    if '|' in gt:
        gt = '|'.join(sorted(gt.split('|')))
    if res is not None:
        if isinstance(res, list):
            res = '|'.join(sorted([str(x) for x in res]))
        elif isinstance(res, pd.Series):
            if len(res) == 1:
                res = str(res.iloc[0])
            else:
                res = '|'.join(sorted([str(x) for x in result.tolist()]))
    return res == gt

In [8]:
def get_accuracy_wtq(data_path,conf_path,tests = None):
    data = load_from_disk(data_path)
    config = load_config(conf_path)
    tests = list(config.keys()) if tests==None else tests
    acc_old = None
    for test in tests:
        print(test)
        process_data = data.filter(lambda x : x[f'{test}_answ_correct'] != 'None',num_proc=17)
        dd = data.filter(lambda x: True if x[f'{test}_label']!='None' else False,num_proc=17)
        print('Процент корректируемых ответов',process_data.shape[0]/data.shape[0]*100)
        cor = process_data.filter(lambda x : True if x[f'{test}_label']==str(bool(x['label'])) else False,num_proc=17)
        print('успешно скорректированные отноосительно корректируемых ',cor.shape[0]/process_data.shape[0]*100)
        print('выполнимость',dd.shape[0]/data.shape[0]*100)
        true = dd.filter(lambda x : True if evaluate(x[f'{test}_label'],x['label']) else False,num_proc=17)
        print('коректность вывыполняемых ',true.shape[0]/dd.shape[0]*100)
        full = data.filter(lambda x : True if evaluate(x[f'{test}_label'],x['label']) else False,num_proc=17)
        acc = full.shape[0]/data.shape[0]*100
        without_corr = data.filter(lambda x : x[f'{test}_answ_correct'] == 'None',num_proc=17)
        full_wo_corr = without_corr.filter(lambda x : True if evaluate(x[f'{test}_label'],x['label']) else False,num_proc=17)
        print('Общая точность',acc)
        print('точность без учета коррекции',full_wo_corr.shape[0]/data.shape[0]*100)
        if acc_old == None:
            acc_old = acc
        print('different',acc-acc_old)
        acc_old = acc

## Бейзлайн wtq

####  чистый deep seek coder 7b 1/5 train prompt

In [11]:
get_accuracy_wtq('wtq_test_xml_baseline','semtab_xml_config_wtq_baseline.yaml')

semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50
Процент корректируемых ответов 14.957363447799032
успешно скорректированные отноосительно корректируемых  0.46224961479198773
выполнимость 86.19497580087577


Filter (num_proc=17):   0%|          | 0/3740 [00:00<?, ? examples/s]

коректность вывыполняемых  35.80213903743316


Filter (num_proc=17):   0%|          | 0/4339 [00:00<?, ? examples/s]

Filter (num_proc=17):   0%|          | 0/4339 [00:00<?, ? examples/s]

Filter (num_proc=17):   0%|          | 0/3690 [00:00<?, ? examples/s]

Общая точность 30.85964507951141
точность без учета коррекции 28.140124452638858
different 0.0


#### deep seek coder 7b 1/5 repanda (tabFact train) train prompt

In [12]:
get_accuracy_wtq('wtq_test_xml_baseline_repanda','semtab_xml_config_wtq_baseline.yaml')

semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50


Filter (num_proc=17):   0%|          | 0/4339 [00:00<?, ? examples/s]

Filter (num_proc=17):   0%|          | 0/4339 [00:00<?, ? examples/s]

Процент корректируемых ответов 1.9359299377736807


Filter (num_proc=17):   0%|          | 0/84 [00:00<?, ? examples/s]

...

коректность вывыполняемых  46.88486936548952


Filter (num_proc=17):   0%|          | 0/4339 [00:00<?, ? examples/s]

Filter (num_proc=17):   0%|          | 0/4339 [00:00<?, ? examples/s]

Filter (num_proc=17):   0%|          | 0/4255 [00:00<?, ? examples/s]

Общая точность 37.635399861719286
точность без учета коррекции 37.128370592302375
different 0.0


In [16]:
data = load_from_disk('wtq_test_xml_baseline_repanda')
data

Dataset({
    features: ['id', 'statement', 'context', 'label', 'table', 'table_text', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_answ', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_label', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_answ_correct', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_last_err'],
    num_rows: 4339
})

In [27]:
full = data.filter(lambda x : False if evaluate(x[f'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_label'],x['label']) else True,num_proc=17)

In [28]:
full

Dataset({
    features: ['id', 'statement', 'context', 'label', 'table', 'table_text', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_answ', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_label', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_answ_correct', 'semtab_xml_attributes_semantic_datatype_exampples_top1_tresh50_last_err'],
    num_rows: 2706
})

In [24]:
full

1633